Title: Model_Outout_test.ipynb

Purpose: Compare the different CMIP ESMs and ERA5 Datasets

Author: Onno Nennecke on 04.05.2025 Modified: 04.05.2025

### Load libraries and functions

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import glob
import time
import pandas as pd

# Importing functions
import Functions.wind_model_func as wind_model_func
import Functions.solar_model_func as solar_model_func
import Functions.demand as demand_func
import Functions.grid_func as grid_func
import Functions.config as config

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


### Load datasets

In [6]:
ERA5_dataset = xr.open_dataset('/climca/people/onennecke/model_output/ERA5_timeseries.nc')
ERA5_dataset

<xarray.Dataset> Size: 386kB
Dimensions:        (time: 4017)
Coordinates:
  * time           (time) datetime64[ns] 32kB 2013-01-01 ... 2023-12-31
    crs            int64 8B ...
    gridtype       <U6 24B ...
    country        float64 8B ...
    period         <U4 16B ...
Data variables: (12/13)
    temp           (time) float64 32kB ...
    demand         (time) float64 32kB ...
    sfcWind        (time) float32 16kB ...
    rsds           (time) float32 16kB ...
    tas            (time) float32 16kB ...
    tasmax         (time) float32 16kB ...
    ...             ...
    wind_on_prod   (time) float64 32kB ...
    solar_prod     (time) float64 32kB ...
    total_prod     (time) float64 32kB ...
    Netto          (time) float64 32kB ...
    Residual_load  (time) float64 32kB ...
    demand_weekly  (time) float64 32kB ...

In [3]:
path = '/climca/people/onennecke/model_output/winter_data/'
files = sorted(glob.glob(os.path.join(path, '*.nc')))
ts_datasets = xr.open_mfdataset(files, combine='nested', concat_dim='ESM_run')
# files
# After loading the dataset
ts_datasets

<xarray.Dataset> Size: 15MB
Dimensions:        (ESM_run: 99, time: 1820)
Coordinates:
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    country        float64 8B 9.0
    period         <U4 16B 'week'
    run            (ESM_run) <U10 4kB 'r1i1p1f1' 'r4i1p1f1' ... 'r9i1p1f2'
    ESM            (ESM_run) <U13 5kB 'ACCESS-CM2' ... 'UKESM1-0-LL'
  * ESM_run        (ESM_run) <U23 9kB 'ACCESS-CM2_r1i1p1f1' ... 'UKESM1-0-LL_...
    winter_year    (time) int64 15kB dask.array<chunksize=(1820,), meta=np.ndarray>
    day_of_winter  (time) int64 15kB dask.array<chunksize=(1820,), meta=np.ndarray>
    winter_season  (time) <U8 58kB dask.array<chunksize=(1820,), meta=np.ndarray>
Data variables:
    temp           (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    demand         (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    sfcWind        (ESM_run, time) float32 721kB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    rsds           (ESM_run, time) float32 721kB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    tas            (ESM_run, time) float32 721kB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    tasmax         (ESM_run, time) float32 721kB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    wind_off_prod  (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    wind_on_prod   (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    solar_prod     (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    total_prod     (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    Netto          (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>
    Residual_load  (ESM_run, time) float64 1MB dask.array<chunksize=(1, 1820), meta=np.ndarray>

### Define used models

In [4]:
# Load climate data

MIP = 'ScenarioMIP' # CMIP
Institution = '*'
# ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg', 'GFDL-CM4', 'GFDL-ESM4', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'MPI-ESM1-2-HR',
#            'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL']
ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'EC-Earth3', 'GFDL-ESM4', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL'] # 'EC-Earth3-Veg'
# ESMs = ['ACCESS-CM2', 'EC-Earth3', 'MPI-ESM1-2-HR', 'MRI-ESM2-0'] # Datetime Format: datetime64
ESMs = ['BCC-CSM2-MR', 'CESM2', 'GFDL-ESM4', 'TaiESM1'] # Datetime Format: cftime.DatetimeNoLeap
# ESMs = ['KACE-1-0-G', 'UKESM1-0-LL'] # Datetime Format: cftime.Datetime360Day
ESMs = ['BCC-CSM2-MR']


# ESMs = ['EC-Earth3'] # 'EC-Earth3-Veg'

# ESMs = ['UKESM1-0-LL']
# ESMs = ['TaiEMS1', 'UKESM1-0LL']
scenario = 'ssp370'
# run = 'r1i1p1f1'
time_res = 'day'
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables
grid_def = '*'
version = '*'


In [5]:
all_ts_outputs = []
for ESM in ESMs:
    print('ESM: ', ESM)
    path = f'/climca/data/CMIP6/{MIP}/{Institution}/{ESM}/{scenario}/'
    matching_dirs = glob.glob(path)
    if len(matching_dirs) == 1:
        runs = os.listdir(matching_dirs[0])
    elif len(matching_dirs) >= 1:
        runs = os.listdir(matching_dirs[0]) + os.listdir(matching_dirs[1])
    else:
        print(f"Found {len(matching_dirs)} matching directories for {ESM} and {scenario}")
        break
    print('Runs: ', runs)
    for run in runs:
        ds_list = [] # List to hold individual datasets (one for each variable)

        run_time = time.time()
        print('Run: ', run)
        run_path = os.path.join(matching_dirs[0], run, 'day/') # Watch out for last model (then manually change to matching_dirs[1])
        
        # Check if all required folders (in `variables`) exist
        missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]
        
        if missing_folders:
            print(f"Missing folders in {run_path}: {missing_folders}")
            continue
        
        missing_data = [
            var for var in variables 
            if not os.path.isdir(os.path.join(run_path, var)) or 
               not any(
                   os.path.isfile(f) 
                   for f in glob.glob(os.path.join(run_path, var, '*', '*', '*'))  # glob pattern to match files two levels down
               )
        ]
        
        if missing_data:
            print(f"Missing data in {run_path}: {missing_data}")
            continue
        
        for variable in variables:
            # print('Variable: ', variable)
            #path =f'/climca/data/CMIP6/CMIP/NCAR/{ESM}/{scenario}/{run}/day/{variable}/gn/v20190308/{variable}_day_{ESM}_{scenario}_{run}_gn_*'
            path = f'/climca/data/CMIP6/{MIP}/{Institution}/{ESM}/{scenario}/{run}/{time_res}/{variable}/{grid_def}/{version}/{variable}_{time_res}_{ESM}_{scenario}_{run}_*'
            # print('Open: ', path)
            
            # Filter out files with extensions after .nc
            files = [f for f in glob.glob(path) if f.endswith('.nc')]
            
            # Open with preprocessing (spatial filtering)
            if files:
                nc = xr.open_mfdataset(files, preprocess=grid_func.preprocess)
            else:
                print("No valid .nc files found!")
            # nc = xr.open_mfdataset(path, preprocess=grid_func.preprocess)
                
        #     # Keep only the desired variable, but retain Dataset structure
            nc = nc[[variable]]
            
            
            # Filter years
            nc = nc.sel(time=nc.time.dt.year.isin(range(2015, 2025)))

        #     # Append to list for later merging
            ds_list.append(nc)
            
            ds_list = [ds.drop_vars('height') if 'height' in ds.coords else ds for ds in ds_list]

        # # Combine all into a single dataset
        clim_ds = xr.merge(ds_list)
                
        if ds_list[2]['tas'].units == 'K':
            clim_ds['tas'] = clim_ds['tas'] - 273.15
        
        if ds_list[3]['tasmax'].units == 'K':
            clim_ds['tasmax'] = clim_ds['tasmax'] - 273.15
            
        # Regrid the combined dataset
        combined_ds = grid_func.regrid(clim_ds)
        
        timeseries_ds = combined_ds.copy()
        timeseries_ds['sfcWind'] = combined_ds['sfcWind'].mean(dim=['lat', 'lon'])
        timeseries_ds['rsds'] = combined_ds['rsds'].mean(dim=['lat', 'lon'])
        timeseries_ds['tas'] = combined_ds['tas'].mean(dim=['lat', 'lon'])
        timeseries_ds['tasmax'] = combined_ds['tasmax'].mean(dim=['lat', 'lon'])
        
        ts_output = timeseries_ds.assign_coords(run = run, ESM = ESM, ESM_run = f'{ESM}_{run}')
        all_ts_outputs.append(ts_output)

for ds in all_ts_outputs:
    ds['time'] = all_ts_outputs[0]['time']
merged_ts = xr.concat(all_ts_outputs, dim='ESM_run')


ESM:  BCC-CSM2-MR
Runs:  ['r1i1p1f1']
Run:  r1i1p1f1


#### Some plots to check everything is alright